### Actividad de Clase y Tarea 
- Jorge Ruiz Esquer 
- martes 20 de octubre 

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.stats import gmean



# Simular variables normales estándar
Distribucion_normal = np.random.normal(0, 1, 10000)


Distribucion_normal_positiva = (Distribucion_normal > 0).mean()
Distribucion_normal_negativa = (Distribucion_normal < 0).mean()
Distribucion_normal_positiva, Distribucion_normal_negativa

(0.4983, 0.5017)

## Calcular el precio de una call 

(esto solo para saber el valor real de nuestra opción y ver nuestra aproximación con el metodo de reducción de varianza)

In [27]:
# Parámetros
S0 = 100
K = 100
sigma = 0.20
r = 0.05
T = 1

M = 10_000
n = 252

dt = T / n


# Generar variables normales aleatorias
Z = np.random.normal(0, 1, (M // 2, n))

# Variables antitéticas
Z_anti = -Z

# Simular precios
def simular_rutas(Z):
    m = Z.shape[0]
    S = np.zeros((m, n + 1))
    S[:, 0] = S0

    for t in range(n):
        S[:, t+1] = S[:, t] * np.exp(
            (r - sigma**2 / 2) * dt
            + sigma * np.sqrt(dt) * Z[:, t]
        )

    return S

# Precio Black-Scholes para comparar
d1 = (np.log(S0 / K) + (r + sigma**2 / 2) * T) / (sigma * np.sqrt(T))
d2 = d1 - sigma * np.sqrt(T)

call_BS = S0 * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)


print("Precio Call Black-Scholes:", call_BS)

Precio Call Black-Scholes: 10.450583572185565


## Tabla Z y -Z 

In [28]:
# Parámetros
S0 = 100
K = 100
sigma = 0.20
r = 0.05
T = 1

# Valores Z dados por el profesor
Z = np.array([
    0.0769,
    -0.2985,
    -0.7983,
    0.9930,
    -1.3465,
    1.2462,
    -1.1405,
    0.7615,
    0.2816,
    0.8100
])

# Precio final del activo
ST_Z = S0 * np.exp(
    (r - sigma**2 / 2) * T + sigma * np.sqrt(T) * Z
)

ST_menos_Z = S0 * np.exp(
    (r - sigma**2 / 2) * T - sigma * np.sqrt(T) * Z
)

# Payoff de la call europea
V_Z = np.maximum(ST_Z - K, 0)
V_menos_Z = np.maximum(ST_menos_Z - K, 0)

# Tabla de resultados
tabla = pd.DataFrame({
    "Z": Z,
    "V_Z": V_Z,
    "V_-Z": V_menos_Z
    
})

print(tabla.round(4))

        Z      V_Z     V_-Z
0  0.0769   4.6425   1.4727
1 -0.2985   0.0000   9.3846
2 -0.7983   0.0000  20.8839
3  0.9930  25.6839   0.0000
4 -1.3465   0.0000  34.8914
5  1.2462  32.2125   0.0000
6 -1.1405   0.0000  29.4468
7  0.7615  19.9974   0.0000
8  0.2816   9.0155   0.0000
9  0.8100  21.1671   0.0000


In [18]:
# Promedio de cada pareja antitética
v_promedio = (V_Z + V_menos_Z) / 2

Z_promedio = V_Z/10

# Agregar a la tabla
tabla["V_promedio"] = v_promedio

# Mostrar cada Z con su promedio
print(tabla[["Z", "V_Z", "V_-Z", "V_promedio"]].round(4))

# Promedio general de todas las parejas
promedio_total = np.mean(v_promedio)

print("\nPromedio de todas las parejas:", round(promedio_total, 4))

        Z      V_Z     V_-Z  V_promedio
0  0.0769   4.6425   1.4727      3.0576
1 -0.2985   0.0000   9.3846      4.6923
2 -0.7983   0.0000  20.8839     10.4419
3  0.9930  25.6839   0.0000     12.8420
4 -1.3465   0.0000  34.8914     17.4457
5  1.2462  32.2125   0.0000     16.1062
6 -1.1405   0.0000  29.4468     14.7234
7  0.7615  19.9974   0.0000      9.9987
8  0.2816   9.0155   0.0000      4.5078
9  0.8100  21.1671   0.0000     10.5835

Promedio de todas las parejas: 10.4399


En esta práctica se aplicó el método de variables antitéticas para estimar el valor de una opción call europea. Se utilizaron diez valores de \(Z\) y sus respectivos opuestos \(-Z\), calculando el payoff de cada escenario y el promedio de cada pareja. El promedio final obtenido fue de 10.4399, cercano al precio teórico de Black-Scholes de 10.4506.

### Parte a completar de tarea 

## Generar 100 rutas con variable de control de opción asiática geométrica.

Utilizar método visto en clase en ejercicio hecho a mano.

In [30]:
# Parámetros
S0 = 100
K = 100
r = 0.05
sigma = 0.20
T = 1

M = 100       # Número de rutas
n = 252       # Días de negociación
dt = T / n


# 1. Generar variables normales
Z = np.random.normal(0, 1, (M, n))

# 2. Simular las rutas de precios
S = np.zeros((M, n + 1))
S[:, 0] = S0

for t in range(n):
    S[:, t+1] = S[:, t] * np.exp(
        (r - sigma**2 / 2) * dt
        + sigma * np.sqrt(dt) * Z[:, t]
    )

# 3. Promedios aritmético y geométrico
precios = S[:, 1:]

A = np.mean(precios, axis=1)
G = np.exp(np.mean(np.log(precios), axis=1))

# 4. Payoffs de las opciones
Y = np.maximum(A - K, 0)  # Call asiática aritmética
X = np.maximum(G - K, 0)  # Call asiática geométrica

# 5. Descontar los payoffs al presente
Y_desc = np.exp(-r * T) * Y
X_desc = np.exp(-r * T) * X

# 6. Precio teórico de la call asiática geométrica
mu_G = np.log(S0) + (r - sigma**2 / 2) * T * (n + 1) / (2 * n)

var_log_G = (
    sigma**2 * T * (n + 1) * (2*n + 1)
    / (6 * n**2)
)

desv_G = np.sqrt(var_log_G)

d2 = (mu_G - np.log(K)) / desv_G
d1 = d2 + desv_G

precio_geo = np.exp(-r*T) * (
    np.exp(mu_G + var_log_G/2) * norm.cdf(d1)
    - K * norm.cdf(d2)
)

# 7. Calcular beta 
beta = np.cov(Y_desc, X_desc, ddof=1)[0, 1] / np.var(X_desc, ddof=1)

# 8. Precio estimado con variable de control
precio_MC = np.mean(Y_desc)

precio_CV = precio_MC - beta * (
    np.mean(X_desc) - precio_geo
)


# 10. Resultado
print("\n--- RESULTADOS FINALES ---")
print("Precio teórico geométrico:", round(precio_geo, 4))
print("Beta:", round(beta, 4))
print("Precio Monte Carlo aritmético:", round(precio_MC, 4))
print("Precio con variable de control:", round(precio_CV, 4))


--- RESULTADOS FINALES ---
Precio teórico geométrico: 5.5655
Beta: 1.0303
Precio Monte Carlo aritmético: 6.1806
Precio con variable de control: 5.7531


En esta práctica se aplicó el método de variable de control para estimar el precio de una opción call asiática aritmética, utilizando como variable de control una opción asiática geométrica. Se realizaron 100 simulaciones y se obtuvo un precio Monte Carlo de 6.1806, mientras que el precio estimado con variable de control fue de 5.7531. El precio teórico de la opción geométrica fue de 5.5655 y el coeficiente beta estimado fue de 1.034221.